# Model vs Student Comparison

Compares 4 AI models (Gemini 2.0 Flash, Gemini 3.0 Flash, GPT-5.4, Claude Sonnet 4.6) against student ground truth (majority vote from 3 annotators) on 8 placental GEO papers.

**Excluded:** GSE130339, GSE155750

In [ ]:
# Cell 1: Load student annotations from all 3 sets
import pandas as pd
import numpy as np
import json
import os
import re

STUDENT_FILE = "manual_review_annotated.xlsx"
EXCLUDE_GEO = {"GSE130339", "GSE155750"}

# Column index -> question name mapping (from row 1 of the Excel)
QUESTION_COLS = {
    4:  "Supervisor/Contact/Corresponding author name",
    5:  "Supervisor/Contact/Corresponding author email",
    6:  "Main topic of the publication",
    7:  "Pregnancy trimester (1st, 2nd, 3rd, term, premature)",
    8:  "Birthweight of offspring provided (yes/no)",
    9:  "Gestational Age at delivery provided (yes/no)",
    10: "GA at delivery (weeks)",
    11: "Gestational Age at sample collection provided (yes/no)",
    12: "GA at sample collection (weeks)",
    13: "Sex of Offspring Provided (yes/no)",
    14: "Parity provided (yes/no)",
    15: "Gravidity provided (yes/no)",
    16: "Number of offspring per pregnancy provided (yes/no)",
    17: "Self-reported race/ethnicity of mother provided (yes/no)",
    18: "Genetic ancestry or genetic strain provided (yes/no)",
    19: "Maternal Height provided (yes/no)",
    20: "Maternal Pre-pregnancy Weight provided (yes/no)",
    21: "Paternal Height provided (yes/no)",
    22: "Paternal Weight provided (yes/no)",
    23: "Maternal age at sample collection provided (yes/no)",
    24: "Paternal age at sample collection provided (yes/no)",
    25: "Samples from pregnancy complications collected",
    26: "Mode of delivery provided (yes/no)",
    27: "Pregnancy complications in data set (list)",
    28: "Fetal complications listed (yes/no)",
    29: "Fetal complications in data set (list)",
    30: "Other Phenotypes Provided (list)",
    31: "Hospital/Center where samples were collected",
    32: "Country where samples were collected",
}

# Load all 3 student sets
student_sets = {}
for sheet in ["Set1", "Set2", "Set3"]:
    df = pd.read_excel(STUDENT_FILE, sheet_name=sheet, header=None)
    records = {}
    for row_idx in range(2, df.shape[0]):
        geo_id = str(df.iloc[row_idx, 0]).strip()
        if geo_id in EXCLUDE_GEO:
            continue
        answers = {}
        for col_idx, q_name in QUESTION_COLS.items():
            val = df.iloc[row_idx, col_idx]
            answers[q_name] = val
        records[geo_id] = answers
    student_sets[sheet] = records
    print(f"{sheet}: {len(records)} papers loaded")

# Get the 8 papers we're working with
PAPERS = sorted(set.intersection(*[set(s.keys()) for s in student_sets.values()]))
print(f"\nWorking with {len(PAPERS)} papers: {PAPERS}")

In [ ]:
# Cell 2: Normalization functions
#
# Students write answers inconsistently: "yes"/"Yes"/"YES", "USA"/"United States",
# nan/"N/A"/"Not Provided", etc. These normalizers bring everything to a common format.

# ── Yes/No normalization ──
def norm_yesno(val):
    """Normalize to 'Yes' or 'No'. Treats nan/blank/ambiguous as 'No'."""
    s = str(val).strip().lower()
    if s in {"nan", "none", "", "n/a", "not provided", "na"}:
        return "No"
    if any(w in s for w in ["yes", "true", "provided", "reported"]):
        return "Yes"
    if s == "no":
        return "No"
    # If they wrote an actual answer (e.g. "hydatidiform mole" for complications),
    # that means the data IS there -> Yes
    if len(s) > 3 and s not in {"false", "none", "n/a"}:
        return "Yes"
    return "No"


# ── Country normalization ──
COUNTRY_ALIASES = {
    "usa": "USA",
    "united states": "USA",
    "united states of america": "USA",
    "us": "USA",
    "u.s.": "USA",
    "u.s.a.": "USA",
    "america": "USA",
    "uk": "UK",
    "united kingdom": "UK",
    "england": "UK",
    "belgium": "Belgium",
    "belgian": "Belgium",
    "canada": "Canada",
    "pakistan": "Pakistan",
    "hungary": "Hungary",
    "china": "China",
    "japan": "Japan",
    "germany": "Germany",
    "france": "France",
    "australia": "Australia",
    "india": "India",
}

def norm_country(val):
    s = str(val).strip()
    if s.lower() in {"nan", "none", "", "n/a", "not provided", "na"}:
        return "Not Provided"
    return COUNTRY_ALIASES.get(s.lower(), s.title())


# ── Trimester normalization ──
TRIMESTER_MAP = {
    "1st": {"1", "1st", "first", "first trimester"},
    "2nd": {"2", "2nd", "second", "second trimester"},
    "3rd": {"3", "3rd", "third", "third trimester"},
    "Term": {"term", "full-term", "full term", "at term"},
    "Premature": {"premature", "preterm", "pre-term", "ptb"},
}

def norm_trimester_single(s):
    s = s.strip().lower()
    for canon, variants in TRIMESTER_MAP.items():
        if s in variants:
            return canon
    if "1" in s or "first" in s: return "1st"
    if "2" in s or "second" in s: return "2nd"
    if "3" in s or "third" in s: return "3rd"
    if "term" in s and "pre" not in s: return "Term"
    if "pre" in s or "premature" in s: return "Premature"
    return None  # unrecognized

def norm_trimester(val):
    s = str(val).strip()
    if s.lower() in {"nan", "none", "", "n/a", "not provided", "na"}:
        return "Not Provided"
    # Split on commas, slashes, "and"
    tokens = re.split(r"[,/]+|\band\b", s.lower())
    results = []
    for tok in tokens:
        tok = tok.strip()
        if not tok:
            continue
        n = norm_trimester_single(tok)
        if n:
            results.append(n)
    if results:
        return ", ".join(sorted(set(results)))
    return "Not Provided"


# ── GA normalization ──
def norm_ga(val):
    s = str(val).strip()
    if s.lower() in {"nan", "none", "", "n/a", "not provided", "na", "no"}:
        return "Not Provided"
    # Extract all numbers
    nums = re.findall(r"\d+\.?\d*", s)
    if nums:
        floats = [float(n) for n in nums]
        # Return mean of extracted numbers as representative value
        return round(np.mean(floats), 1)
    return "Not Provided"


# ── List normalization (complications, phenotypes) ──
def norm_list(val):
    s = str(val).strip()
    if s.lower() in {"nan", "none", "", "n/a", "not provided", "na", "no", "[]"}:
        return []
    items = re.split(r"[;,\n]+", s)
    cleaned = []
    for item in items:
        item = item.strip().lower()
        if item and item not in {"nan", "none", "n/a", "na"}:
            cleaned.append(item)
    return sorted(set(cleaned))


# ── Free text normalization ──
def norm_freetext(val):
    s = str(val).strip()
    if s.lower() in {"nan", "none", "", "n/a", "not provided", "na", "unknown"}:
        return "Not Provided"
    return s


# ── Question type classification ──
YESNO_QUESTIONS = [
    "Birthweight of offspring provided (yes/no)",
    "Gestational Age at delivery provided (yes/no)",
    "Gestational Age at sample collection provided (yes/no)",
    "Sex of Offspring Provided (yes/no)",
    "Parity provided (yes/no)",
    "Gravidity provided (yes/no)",
    "Number of offspring per pregnancy provided (yes/no)",
    "Self-reported race/ethnicity of mother provided (yes/no)",
    "Genetic ancestry or genetic strain provided (yes/no)",
    "Maternal Height provided (yes/no)",
    "Maternal Pre-pregnancy Weight provided (yes/no)",
    "Paternal Height provided (yes/no)",
    "Paternal Weight provided (yes/no)",
    "Maternal age at sample collection provided (yes/no)",
    "Paternal age at sample collection provided (yes/no)",
    "Samples from pregnancy complications collected",
    "Mode of delivery provided (yes/no)",
    "Fetal complications listed (yes/no)",
]

GA_QUESTIONS = [
    "GA at delivery (weeks)",
    "GA at sample collection (weeks)",
]

LIST_QUESTIONS = [
    "Pregnancy complications in data set (list)",
    "Fetal complications in data set (list)",
    "Other Phenotypes Provided (list)",
]

TRIMESTER_QUESTION = "Pregnancy trimester (1st, 2nd, 3rd, term, premature)"
COUNTRY_QUESTION = "Country where samples were collected"


def normalize_answer(question, val):
    """Route to the right normalizer based on question type."""
    if question in YESNO_QUESTIONS:
        return norm_yesno(val)
    elif question == TRIMESTER_QUESTION:
        return norm_trimester(val)
    elif question in GA_QUESTIONS:
        return norm_ga(val)
    elif question in LIST_QUESTIONS:
        return norm_list(val)
    elif question == COUNTRY_QUESTION:
        return norm_country(val)
    else:
        return norm_freetext(val)

print("Normalizers ready.")
print(f"  Yes/No questions: {len(YESNO_QUESTIONS)}")
print(f"  GA questions: {len(GA_QUESTIONS)}")
print(f"  List questions: {len(LIST_QUESTIONS)}")
print(f"  Other: trimester, country, free text")

In [ ]:
# Cell 3: Normalize student answers & build ground truth via majority vote
from collections import Counter

# Normalize all 3 student sets
student_normalized = {}
for set_name, records in student_sets.items():
    norm_records = {}
    for geo_id, answers in records.items():
        norm_answers = {}
        for q, val in answers.items():
            norm_answers[q] = normalize_answer(q, val)
        norm_records[geo_id] = norm_answers
    student_normalized[set_name] = norm_records

# Build ground truth: majority vote across 3 students
ground_truth = {}
student_agreement = {}  # track how often students agree

for geo_id in PAPERS:
    ground_truth[geo_id] = {}
    student_agreement[geo_id] = {}
    
    for q in QUESTION_COLS.values():
        vals = []
        for set_name in ["Set1", "Set2", "Set3"]:
            v = student_normalized[set_name].get(geo_id, {}).get(q, "Not Provided")
            vals.append(v)
        
        if q in YESNO_QUESTIONS:
            # Majority vote: 2/3 agree
            yes_count = sum(1 for v in vals if v == "Yes")
            gt = "Yes" if yes_count >= 2 else "No"
            agree = 3 if (yes_count == 3 or yes_count == 0) else 2
        
        elif q == TRIMESTER_QUESTION:
            counts = Counter(str(v) for v in vals)
            gt = counts.most_common(1)[0][0]
            agree = counts.most_common(1)[0][1]
        
        elif q in GA_QUESTIONS:
            # If 2+ have numeric values, take median. Otherwise "Not Provided".
            numeric = [v for v in vals if isinstance(v, (int, float))]
            if len(numeric) >= 2:
                gt = round(float(np.median(numeric)), 1)
                # Agree if all numeric values within 2 weeks
                agree = 3 if (max(numeric) - min(numeric) <= 2) else 2
            elif len(numeric) == 1:
                gt = numeric[0]
                agree = 1
            else:
                gt = "Not Provided"
                agree = sum(1 for v in vals if v == "Not Provided")
        
        elif q in LIST_QUESTIONS:
            # Union of all items mentioned by any student
            all_items = set()
            for v in vals:
                if isinstance(v, list):
                    all_items.update(v)
            gt = sorted(all_items)
            # Agreement = how many students listed at least 1 item (or all empty)
            non_empty = sum(1 for v in vals if isinstance(v, list) and len(v) > 0)
            agree = 3 if (non_empty == 3 or non_empty == 0) else non_empty
        
        elif q == COUNTRY_QUESTION:
            counts = Counter(str(v) for v in vals)
            gt = counts.most_common(1)[0][0]
            agree = counts.most_common(1)[0][1]
        
        else:  # free text
            # Use most common, or first non-empty
            str_vals = [str(v) for v in vals if str(v) != "Not Provided"]
            if str_vals:
                counts = Counter(str_vals)
                gt = counts.most_common(1)[0][0]
                agree = counts.most_common(1)[0][1]
            else:
                gt = "Not Provided"
                agree = 3
        
        ground_truth[geo_id][q] = gt
        student_agreement[geo_id][q] = agree

# Summary
total_qs = len(PAPERS) * len(QUESTION_COLS)
full_agree = sum(1 for geo in PAPERS for q in QUESTION_COLS.values() 
                 if student_agreement[geo][q] == 3)
two_agree = sum(1 for geo in PAPERS for q in QUESTION_COLS.values() 
                if student_agreement[geo][q] == 2)
no_agree = total_qs - full_agree - two_agree

print(f"Ground truth built for {len(PAPERS)} papers x {len(QUESTION_COLS)} questions = {total_qs} total")
print(f"\nStudent agreement:")
print(f"  3/3 agree: {full_agree} ({100*full_agree/total_qs:.0f}%)")
print(f"  2/3 agree: {two_agree} ({100*two_agree/total_qs:.0f}%)")
print(f"  No majority: {no_agree} ({100*no_agree/total_qs:.0f}%)")

# Show yes/no agreement specifically
yn_total = len(PAPERS) * len(YESNO_QUESTIONS)
yn_full = sum(1 for geo in PAPERS for q in YESNO_QUESTIONS 
              if student_agreement[geo][q] == 3)
print(f"\nYes/No questions: {yn_full}/{yn_total} ({100*yn_full/yn_total:.0f}%) unanimous")

## Student Annotation Analysis

Before comparing models, let's understand the student data: where do they agree, where do they disagree, and how messy are the raw answers?

In [ ]:
# Cell 4a: Raw student answers side-by-side — Yes/No questions (BEFORE normalization)
# Shows exactly what each student typed so you can see the inconsistency

print("YES/NO QUESTIONS — Where Students Disagree (after normalization)")
print("=" * 110)
print(f"\n{'GEO ID':<12s} {'Question':<45s} {'Set1 raw':<12s} {'Set2 raw':<12s} {'Set3 raw':<12s} Normalized")
print("-" * 110)

raw_disagree_count = 0
for geo_id in PAPERS:
    for q in YESNO_QUESTIONS:
        norm_vals = [student_normalized[sn][geo_id][q] for sn in ["Set1", "Set2", "Set3"]]
        
        if len(set(norm_vals)) > 1:  # students disagree even after normalization
            raw_disagree_count += 1
            raw_vals = []
            for sn in ["Set1", "Set2", "Set3"]:
                rv = str(student_sets[sn][geo_id][q]).strip()
                raw_vals.append(rv if rv not in {"nan", ""} else "-")
            
            short_q = q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")[:43]
            gt = ground_truth[geo_id][q]
            print(f"{geo_id:<12s} {short_q:<45s} {raw_vals[0]:<12s} {raw_vals[1]:<12s} {raw_vals[2]:<12s} GT={gt}")

yn_total = len(PAPERS) * len(YESNO_QUESTIONS)
yn_agree = yn_total - raw_disagree_count
print(f"\nYes/No agreement: {yn_agree}/{yn_total} ({100*yn_agree/yn_total:.0f}%) — {raw_disagree_count} disagreements")

In [ ]:
# Student Analysis 2: Per-question agreement rate
# Which questions do students find hardest to agree on?

print("Per-Question Student Agreement (Yes/No)")
print("=" * 75)

q_agree_rates = []
for q in YESNO_QUESTIONS:
    agrees = [student_agreement[geo][q] for geo in PAPERS]
    unanimous = sum(1 for a in agrees if a == 3)
    rate = unanimous / len(agrees)
    q_agree_rates.append((q, rate, unanimous, len(agrees)))

q_agree_rates.sort(key=lambda x: x[1])  # hardest first

for q, rate, unan, total in q_agree_rates:
    short_q = q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")[:50]
    bar = "*" * int(rate * 10)
    print(f"  {rate:5.0%}  {bar:<10s}  {short_q} ({unan}/{total} unanimous)")

print(f"\nMost controversial questions are at the top.")

In [ ]:
# Student Analysis 3: Per-paper agreement rate
# Which papers are hardest for students?

print("Per-Paper Student Agreement (Yes/No)")
print("=" * 60)

paper_rates = []
for geo_id in PAPERS:
    agrees = [student_agreement[geo_id][q] for q in YESNO_QUESTIONS]
    unanimous = sum(1 for a in agrees if a == 3)
    rate = unanimous / len(agrees)
    paper_rates.append((geo_id, rate, unanimous, len(agrees)))

paper_rates.sort(key=lambda x: x[1])  # hardest first

for geo_id, rate, unan, total in paper_rates:
    bar = "*" * int(rate * 20)
    print(f"  {geo_id:<12s} {rate:5.0%}  {bar:<20s}  ({unan}/{total} unanimous)")

print(f"\nPapers with most student disagreement are at the top.")

In [ ]:
# Student Analysis 4: Non-yes/no fields — show raw answers
# Trimester, GA, Country, and List answers where students differ

TRICKY_QUESTIONS = [TRIMESTER_QUESTION, COUNTRY_QUESTION] + GA_QUESTIONS + LIST_QUESTIONS

print("NON-YES/NO FIELDS — Student Raw Answers")
print("=" * 130)

for q in TRICKY_QUESTIONS:
    short_q = q[:55]
    print(f"\n--- {short_q} ---")
    
    for geo_id in PAPERS:
        raw_vals = []
        norm_vals = []
        for sn in ["Set1", "Set2", "Set3"]:
            rv = str(student_sets[sn][geo_id][q]).strip()
            raw_vals.append(rv if rv not in {"nan", ""} else "-")
            norm_vals.append(str(student_normalized[sn][geo_id][q]))
        
        all_same_norm = len(set(norm_vals)) == 1
        marker = "  " if all_same_norm else ">>" 
        gt_val = str(ground_truth[geo_id][q])
        
        print(f"  {marker} {geo_id:<11s}", end="")
        for i, rv in enumerate(raw_vals):
            rv_short = rv[:30]
            print(f"  S{i+1}: {rv_short:<32s}", end="")
        print(f"  GT: {gt_val[:40]}")

In [ ]:
# Student Analysis 5: Pairwise agreement + agreement heatmap
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Pairwise agreement between student sets
SETS = ["Set1", "Set2", "Set3"]
pair_agree = {}
for i, s1 in enumerate(SETS):
    for j, s2 in enumerate(SETS):
        match = total = 0
        for geo_id in PAPERS:
            for q in YESNO_QUESTIONS:
                match += int(student_normalized[s1][geo_id][q] == student_normalized[s2][geo_id][q])
                total += 1
        pair_agree[(s1, s2)] = match / total

print("Pairwise Student Agreement (Yes/No questions):")
print(f"{\"\":<8s}", end="")
for s in SETS:
    print(f"  {s:>8s}", end="")
print()
for s1 in SETS:
    print(f"{s1:8s}", end="")
    for s2 in SETS:
        print(f"  {pair_agree[(s1,s2)]:>7.1%}", end="")
    print()

# Agreement heatmap: papers x questions
fig, ax = plt.subplots(figsize=(14, 9))

matrix = []
q_labels = []
for q in YESNO_QUESTIONS:
    row = [student_agreement[geo][q] for geo in PAPERS]
    matrix.append(row)
    q_labels.append(q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")[:40])

matrix = np.array(matrix)
cmap = ListedColormap(["#e74c3c", "#f39c12", "#2ecc71"])
im = ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=1, vmax=3)

ax.set_xticks(range(len(PAPERS)))
ax.set_xticklabels(PAPERS, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(q_labels)))
ax.set_yticklabels(q_labels, fontsize=9)

for i in range(len(q_labels)):
    for j in range(len(PAPERS)):
        ax.text(j, i, f"{matrix[i,j]}/3", ha="center", va="center", fontsize=8)

ax.set_title("Student Agreement per Question per Paper\n(Green=3/3, Yellow=2/3, Red=no consensus)", fontsize=13)
plt.tight_layout()
os.makedirs("benchmark_results", exist_ok=True)
plt.savefig("benchmark_results/student_agreement_heatmap.png", dpi=300)
plt.show()
print("Saved: benchmark_results/student_agreement_heatmap.png")

---
## Part 2: Model Comparison

Now load the 4 AI models and compare against student ground truth.

In [ ]:
# Cell 4: Load all 4 model results

BENCHMARK_DIR = "benchmark_results"

# New models (from benchmark notebook)
NEW_MODELS = ["gemini-3.0-flash-preview", "gpt-5.4", "claude-sonnet-4.6"]

model_results = {}

# Load 3 new models from benchmark_results/
for model_name in NEW_MODELS:
    model_results[model_name] = {}
    model_dir = os.path.join(BENCHMARK_DIR, model_name)
    for geo_id in PAPERS:
        fpath = os.path.join(model_dir, f"{geo_id}.json")
        if os.path.exists(fpath):
            with open(fpath) as f:
                raw = json.load(f)
            # Normalize: extract answer from {answer, confidence, reasoning} structure
            normed = {}
            for q in QUESTION_COLS.values():
                entry = raw.get(q, {})
                if isinstance(entry, dict):
                    raw_ans = entry.get("answer", "Not Provided")
                else:
                    raw_ans = entry
                normed[q] = normalize_answer(q, raw_ans)
            model_results[model_name][geo_id] = normed
    print(f"{model_name}: {len(model_results[model_name])}/8 papers")

# Load old Gemini 2.0 Flash from ai.xlsx
old_ai = pd.read_excel("ai.xlsx")
geo_col = old_ai.columns[1]  # 'GEO Series ID (GSE___)'

# Map ai.xlsx columns to our question names
AI_COL_MAP = {
    "Supervisor/Contact/Corresponding author name": "Supervisor/Contact/Corresponding author name",
    "Supervisor/Contact/Corresponding author email": "Supervisor/Contact/Corresponding author email",
    "Main topic of the publication": "Main topic of the publication",
    "Pregnancy trimester": "Pregnancy trimester (1st, 2nd, 3rd, term, premature)",
    "Birthweight of offspring provided (yes/no)": "Birthweight of offspring provided (yes/no)",
    "Gestational Age at delivery provided (yes/no)": "Gestational Age at delivery provided (yes/no)",
    "GA at delivery (weeks)": "GA at delivery (weeks)",
    "Gestational Age at sample collection provided (yes/no)": "Gestational Age at sample collection provided (yes/no)",
    "GA at sample collection (weeks)": "GA at sample collection (weeks)",
    "Sex of Offspring Provided (yes/no)": "Sex of Offspring Provided (yes/no)",
    "Parity provided (yes/no)": "Parity provided (yes/no)",
    "Gravidity provided (yes/no)": "Gravidity provided (yes/no)",
    "Number of offspring per pregnancy provided (yes/no)": "Number of offspring per pregnancy provided (yes/no)",
    "Self-reported race/ethnicity of mother provided (yes/no)": "Self-reported race/ethnicity of mother provided (yes/no)",
    "Genetic ancestry or genetic strain provided (yes/no)": "Genetic ancestry or genetic strain provided (yes/no)",
    "Maternal Height provided (yes/no)": "Maternal Height provided (yes/no)",
    "Maternal Pre-pregnancy Weight provided (yes/no)": "Maternal Pre-pregnancy Weight provided (yes/no)",
    "Paternal Height provided (yes/no)": "Paternal Height provided (yes/no)",
    "Paternal Weight provided (yes/no)": "Paternal Weight provided (yes/no)",
    "Maternal age at sample collection provided (yes/no)": "Maternal age at sample collection provided (yes/no)",
    "Paternal age at sample collection provided (yes/no)": "Paternal age at sample collection provided (yes/no)",
    "Samples from pregnancy complications collected": "Samples from pregnancy complications collected",
    "Mode of delivery provided (yes/no)": "Mode of delivery provided (yes/no)",
    "Pregnancy complications in data set (list)": "Pregnancy complications in data set (list)",
    "Fetal complications listed (yes/no)": "Fetal complications listed (yes/no)",
    "Fetal complications in data set (list)": "Fetal complications in data set (list)",
    "Other Phenotypes Provided (list)": "Other Phenotypes Provided (list)",
    "Hospital/Center where samples were collected": "Hospital/Center where samples were collected",
    "Country where samples were collected": "Country where samples were collected",
}

model_results["gemini-2.0-flash"] = {}
for geo_id in PAPERS:
    row = old_ai[old_ai[geo_col].astype(str).str.strip() == geo_id]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    normed = {}
    for ai_col, q_name in AI_COL_MAP.items():
        if ai_col in old_ai.columns:
            raw_val = row[ai_col]
            normed[q_name] = normalize_answer(q_name, raw_val)
        else:
            normed[q_name] = "Not Provided"
    model_results["gemini-2.0-flash"][geo_id] = normed

print(f"gemini-2.0-flash: {len(model_results['gemini-2.0-flash'])}/8 papers")

ALL_MODELS = ["gemini-2.0-flash"] + NEW_MODELS
print(f"\nAll models: {ALL_MODELS}")

In [ ]:
# Cell 5: Model accuracy vs student ground truth (Yes/No questions only)

accuracy_data = []

for model_name in ALL_MODELS:
    correct = 0
    total = 0
    per_question = {}
    
    for geo_id in PAPERS:
        gt = ground_truth[geo_id]
        pred = model_results.get(model_name, {}).get(geo_id, {})
        
        for q in YESNO_QUESTIONS:
            gt_val = gt.get(q, "No")
            pred_val = pred.get(q, "No")
            match = (gt_val == pred_val)
            correct += int(match)
            total += 1
            
            if q not in per_question:
                per_question[q] = {"correct": 0, "total": 0}
            per_question[q]["correct"] += int(match)
            per_question[q]["total"] += 1
    
    acc = correct / total if total > 0 else 0
    accuracy_data.append({
        "model": model_name,
        "correct": correct,
        "total": total,
        "accuracy": acc,
        "per_question": per_question,
    })
    print(f"{model_name:30s}: {correct}/{total} = {acc:.1%}")

print("\n(Accuracy = match with student majority vote on yes/no questions)")

In [ ]:
# Cell 6: Per-question accuracy across all models (which questions are hard?)
import matplotlib.pyplot as plt

# Build per-question accuracy table
q_acc = {}
for q in YESNO_QUESTIONS:
    q_acc[q] = {}
    for entry in accuracy_data:
        pq = entry["per_question"][q]
        q_acc[q][entry["model"]] = pq["correct"] / pq["total"]

# Also compute student agreement rate per question
q_student_agree = {}
for q in YESNO_QUESTIONS:
    agrees = [student_agreement[geo][q] for geo in PAPERS]
    q_student_agree[q] = sum(1 for a in agrees if a == 3) / len(agrees)

# Sort questions by average model accuracy (hardest first)
q_avg = {q: np.mean(list(accs.values())) for q, accs in q_acc.items()}
sorted_qs = sorted(q_avg.keys(), key=lambda q: q_avg[q])

# Print table
print(f"{'Question':<55s}", end="")
for m in ALL_MODELS:
    short = m.replace("gemini-", "G").replace("claude-sonnet-", "C").replace("gpt-", "GPT")
    print(f"  {short:>8s}", end="")
print(f"  {'Students':>8s}")
print("-" * 110)

for q in sorted_qs:
    short_q = q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")[:53]
    print(f"{short_q:<55s}", end="")
    for m in ALL_MODELS:
        print(f"  {q_acc[q][m]:>7.0%}", end="")
    print(f"  {q_student_agree[q]:>7.0%}")

In [ ]:
# Cell 7: Accuracy bar chart — all models vs ground truth

fig, ax = plt.subplots(figsize=(10, 5))

models = [e["model"] for e in accuracy_data]
accs = [e["accuracy"] for e in accuracy_data]
colors = ["#95a5a6", "#3498db", "#e67e22", "#9b59b6"]

bars = ax.bar(models, accs, color=colors, edgecolor="black", linewidth=0.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{acc:.1%}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("Accuracy vs Student Ground Truth", fontsize=12)
ax.set_title("Model Accuracy on Yes/No Questions\n(vs majority vote of 3 student annotators, 8 papers)", fontsize=13)
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color="green", linestyle="--", alpha=0.3, label="Perfect")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(BENCHMARK_DIR, "accuracy_bar.png"), dpi=300)
plt.show()

In [ ]:
# Cell 8: Pairwise inter-model agreement matrix (yes/no questions)

n_models = len(ALL_MODELS)
agree_matrix = np.zeros((n_models, n_models))

for i, m1 in enumerate(ALL_MODELS):
    for j, m2 in enumerate(ALL_MODELS):
        match = 0
        total = 0
        for geo_id in PAPERS:
            r1 = model_results.get(m1, {}).get(geo_id, {})
            r2 = model_results.get(m2, {}).get(geo_id, {})
            for q in YESNO_QUESTIONS:
                v1 = r1.get(q, "No")
                v2 = r2.get(q, "No")
                match += int(v1 == v2)
                total += 1
        agree_matrix[i, j] = match / total if total > 0 else 0

# Print matrix
print("Pairwise Agreement Matrix (Yes/No questions):")
print(f"{'':25s}", end="")
for m in ALL_MODELS:
    print(f"  {m[:12]:>12s}", end="")
print()
for i, m in enumerate(ALL_MODELS):
    print(f"{m:25s}", end="")
    for j in range(n_models):
        print(f"  {agree_matrix[i,j]:>11.1%}", end="")
    print()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(agree_matrix, cmap="YlGn", vmin=0.7, vmax=1.0)

ax.set_xticks(range(n_models))
ax.set_xticklabels(ALL_MODELS, rotation=30, ha="right", fontsize=10)
ax.set_yticks(range(n_models))
ax.set_yticklabels(ALL_MODELS, fontsize=10)

for i in range(n_models):
    for j in range(n_models):
        ax.text(j, i, f"{agree_matrix[i,j]:.0%}",
                ha="center", va="center", fontsize=12, fontweight="bold")

ax.set_title("Pairwise Model Agreement on Yes/No Questions", fontsize=13)
plt.colorbar(im, label="Agreement Rate")
plt.tight_layout()
plt.savefig(os.path.join(BENCHMARK_DIR, "agreement_matrix.png"), dpi=300)
plt.show()

In [ ]:
# Cell 9: Per-question difficulty line chart
# X-axis = questions, Y-axis = accuracy, one line per model

fig, ax = plt.subplots(figsize=(16, 7))

# Sort by average accuracy (hardest left, easiest right)
x = range(len(sorted_qs))
short_labels = [q.replace(" provided (yes/no)", "").replace(" (yes/no)", "")
                .replace("Samples from pregnancy complications collected", "Preg. complications collected")
                [:40] for q in sorted_qs]

colors = {"gemini-2.0-flash": "#95a5a6", "gemini-3.0-flash-preview": "#3498db",
          "gpt-5.4": "#e67e22", "claude-sonnet-4.6": "#9b59b6"}
markers = {"gemini-2.0-flash": "s", "gemini-3.0-flash-preview": "D",
           "gpt-5.4": "o", "claude-sonnet-4.6": "^"}

for m in ALL_MODELS:
    accs = [q_acc[q][m] for q in sorted_qs]
    ax.plot(x, accs, marker=markers[m], label=m, color=colors[m],
            linewidth=2, markersize=7, alpha=0.85)

# Also plot student agreement as grey dashed
student_line = [q_student_agree[q] for q in sorted_qs]
ax.plot(x, student_line, marker="x", label="Student unanimity",
        color="grey", linewidth=1.5, linestyle="--", alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=55, ha="right", fontsize=8)
ax.set_ylabel("Accuracy vs Ground Truth", fontsize=12)
ax.set_title("Per-Question Accuracy by Model\n(sorted hardest → easiest)", fontsize=13)
ax.set_ylim(-0.05, 1.15)
ax.axhline(y=1.0, color="green", linestyle=":", alpha=0.2)
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BENCHMARK_DIR, "per_question_accuracy.png"), dpi=300)
plt.show()

In [ ]:
# Cell 10: Gemini improvement — old 2.0 Flash vs new 3.0 Flash

print("Gemini 2.0 Flash vs 3.0 Flash Preview — Answer Changes")
print("=" * 70)

flips = []
for geo_id in PAPERS:
    old = model_results.get("gemini-2.0-flash", {}).get(geo_id, {})
    new = model_results.get("gemini-3.0-flash-preview", {}).get(geo_id, {})
    gt = ground_truth[geo_id]
    
    for q in YESNO_QUESTIONS:
        old_v = old.get(q, "No")
        new_v = new.get(q, "No")
        gt_v = gt.get(q, "No")
        
        if old_v != new_v:
            old_correct = (old_v == gt_v)
            new_correct = (new_v == gt_v)
            if new_correct and not old_correct:
                direction = "FIXED"
            elif old_correct and not new_correct:
                direction = "REGRESSED"
            else:
                direction = "CHANGED"
            flips.append({
                "geo_id": geo_id,
                "question": q.replace(" provided (yes/no)", "")[:50],
                "old": old_v,
                "new": new_v,
                "ground_truth": gt_v,
                "direction": direction,
            })

if flips:
    flip_df = pd.DataFrame(flips)
    fixed = (flip_df["direction"] == "FIXED").sum()
    regressed = (flip_df["direction"] == "REGRESSED").sum()
    changed = (flip_df["direction"] == "CHANGED").sum()
    print(f"\nTotal flips: {len(flips)}")
    print(f"  FIXED (wrong→right):     {fixed}")
    print(f"  REGRESSED (right→wrong): {regressed}")
    print(f"  CHANGED (both wrong):    {changed}")
    print(f"\nNet improvement: {fixed - regressed:+d} questions")
    print()
    print(flip_df.to_string(index=False))
else:
    print("No changes between old and new Gemini!")

In [ ]:
# Cell 11: Disagreement deep-dive — show every question where models disagree

print("All Yes/No Disagreements (at least one model differs)")
print("=" * 90)

disagree_rows = []
for geo_id in PAPERS:
    for q in YESNO_QUESTIONS:
        answers = {}
        for m in ALL_MODELS:
            answers[m] = model_results.get(m, {}).get(geo_id, {}).get(q, "No")
        
        vals = list(answers.values())
        if len(set(vals)) > 1:  # at least one disagrees
            gt_val = ground_truth[geo_id].get(q, "No")
            row = {
                "GEO_ID": geo_id,
                "Question": q.replace(" provided (yes/no)", "")[:45],
                "Ground Truth": gt_val,
            }
            for m in ALL_MODELS:
                mark = "*" if answers[m] != gt_val else " "
                short_m = m.replace("gemini-", "G").replace("claude-sonnet-", "C").replace("gpt-", "GPT")
                row[short_m] = f"{answers[m]}{mark}"
            disagree_rows.append(row)

if disagree_rows:
    disagree_df = pd.DataFrame(disagree_rows)
    print(f"\n{len(disagree_rows)} disagreements found (* = wrong vs ground truth)\n")
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.width', 200)
    print(disagree_df.to_string(index=False))
else:
    print("All models agree on every yes/no question!")

In [ ]:
# Cell 12: Save everything to Excel

output_file = os.path.join(BENCHMARK_DIR, "model_vs_student_comparison.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    # Sheet 1: Overall accuracy
    acc_df = pd.DataFrame([{"Model": e["model"], "Correct": e["correct"],
                            "Total": e["total"], "Accuracy": f"{e['accuracy']:.1%}"}
                           for e in accuracy_data])
    acc_df.to_excel(writer, sheet_name="Overall Accuracy", index=False)
    
    # Sheet 2: Per-question accuracy
    pq_rows = []
    for q in sorted_qs:
        row = {"Question": q, "Student Unanimity": f"{q_student_agree[q]:.0%}"}
        for m in ALL_MODELS:
            row[m] = f"{q_acc[q][m]:.0%}"
        pq_rows.append(row)
    pd.DataFrame(pq_rows).to_excel(writer, sheet_name="Per-Question Accuracy", index=False)
    
    # Sheet 3: Full comparison (all questions, all models + ground truth)
    full_rows = []
    for geo_id in PAPERS:
        for q in QUESTION_COLS.values():
            row = {
                "GEO_ID": geo_id,
                "Question": q,
                "Ground Truth (students)": str(ground_truth[geo_id].get(q, "")),
                "Student Agreement": f"{student_agreement[geo_id][q]}/3",
            }
            for m in ALL_MODELS:
                row[m] = str(model_results.get(m, {}).get(geo_id, {}).get(q, ""))
            full_rows.append(row)
    pd.DataFrame(full_rows).to_excel(writer, sheet_name="Full Comparison", index=False)
    
    # Sheet 4: Disagreements
    if disagree_rows:
        pd.DataFrame(disagree_rows).to_excel(writer, sheet_name="Disagreements", index=False)
    
    # Sheet 5: Gemini improvement
    if flips:
        flip_df.to_excel(writer, sheet_name="Gemini Improvement", index=False)

print(f"Saved: {output_file}")